[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P08/P08-1_Regrouper_Joindre.ipynb)

# Regroupements et jointures

Chapitre 8 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Vos organes ne se valent pas : certains tournent à chaud, d'autres à froid, et vous voudriez savoir si leur durée de vie diffère d'un régime à l'autre. Vos pannes sont dans deux fichiers, l'un des mesures, l'autre des modes de défaillance, et il faut les réunir sans perdre ni dupliquer un seul relevé. Un tableur y arrive à la main, une fois ; vous voudriez le faire en quelques lignes, refaire le calcul à chaque nouveau fichier, et savoir que la fusion n'a rien déformé.

**Objectif terminal.** Ce chapitre montre comment regrouper, croiser, recoder et joindre des tableaux avec pandas.

**Objectif instrumental.**

- Regrouper les lignes d'un tableau et calculer une statistique par groupe.
- Croiser deux variables en tableau d'effectifs, pivoter un tableau et le remettre en colonne.
- Recoder des valeurs et convertir des unités colonne par colonne.
- Joindre deux tableaux par une clé, et valider la fusion avant de l'employer.
- Empiler des tableaux de même forme.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Le chapitre 7, après une filière scientifique.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-RUL.xlsx` | 240 | 6 | couple en N·m, puissance en kW, masse en kg, temps de réponse en ms, durée de vie résiduelle en jours |
| `MecaIA-AI4I_maintenance_TypePanne.xlsx` | 10000 | 7 | temps en jours, indicateurs 0 ou 1 |
| `MecaIA-AI4I_maintenance.xlsx` | 10000 | 9 | temps en jours, températures en °C, vitesse en tr/min, couple en N·m, usure en h, puissance en W, vibration en mm/s |
| `MecaIA-TimeSeries_RUL.xlsx` | 18855 | 10 | cycles en heures, charge en %, vitesse en tr/min, température en K, pression en bar, vibration en mm/s, courant en A, contamination en ppm |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 8.1 | Durée de vie par régime thermique | 15 min |
| 8.2 | Pannes par mode de panne | 15 min |
| 8.3 | Fusion des deux fichiers de pannes (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [ ]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-RUL.xlsx",
    "MecaIA-AI4I_maintenance_TypePanne.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance_TypePanne.xlsx",
    "MecaIA-AI4I_maintenance.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance.xlsx",
    "MecaIA-TimeSeries_RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-TimeSeries_RUL.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("pandas", version("pandas"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [ ]:
import pandas as pd
CHEMIN_RUL = DONNEES / "MecaIA-RUL.xlsx"
CHEMIN_MODES = DONNEES / "MecaIA-AI4I_maintenance_TypePanne.xlsx"
CHEMIN_PANNES = DONNEES / "MecaIA-AI4I_maintenance.xlsx"
CHEMIN_FLOTTE = DONNEES / "MecaIA-TimeSeries_RUL.xlsx"
pd.set_option("display.width", 100)

### Section 8.1 : un regroupement répond à une question de comparaison.

In [ ]:
data = pd.read_excel(CHEMIN_RUL)
df = data.copy()
niveau = "niveau_sollicitation_thermique_cat"
# notion: groupby
par_niveau = df.groupby(niveau)["RUL_jours"].agg(["count", "mean", "std"]).round(1)
print(par_niveau)
flotte = pd.read_excel(CHEMIN_FLOTTE)
lignes_par_machine = flotte.groupby("ID_Machine").size()
print(f"flotte : {flotte.shape[0]} lignes, {lignes_par_machine.shape[0]} machines, "
      f"de {lignes_par_machine.min()} à {lignes_par_machine.max()} lignes par machine")
assert (
    flotte.shape == (18855, 10)
), "la fiche du jeu dit la même taille"
assert (
    lignes_par_machine.shape[0] == 100 and lignes_par_machine.min() == 50 and lignes_par_machine.max() == 295
)

In [ ]:
assert par_niveau["count"].sum() == 237, "les lignes sans niveau n'entrent dans aucun groupe"
# notion: agg-nomme
bilan = df.groupby(niveau).agg(duree_moyenne=("RUL_jours", "mean"), effectif=("RUL_jours", "count"),
                               masse_max=("masse_vibrante_kg", "max"))
moyenne_du_groupe = df.groupby(niveau)["RUL_jours"].transform("mean")      # une valeur par ligne
ecart_au_groupe = df["RUL_jours"] - moyenne_du_groupe
assert list(bilan.columns) == ["duree_moyenne", "effectif", "masse_max"]
assert len(moyenne_du_groupe) == len(df), "transform rend autant de lignes que le tableau"
assert (
    abs(ecart_au_groupe.groupby(df[niveau]).sum()).max() < 1e-9
), "écarts centrés dans chaque groupe"
vib_centree = flotte["Vib_RMS_AxeZ_mm_s"] - flotte.groupby("ID_Machine")["Vib_RMS_AxeZ_mm_s"].transform("mean")
assert abs(vib_centree.groupby(flotte["ID_Machine"]).mean()).max() < 1e-9

## Exercice 8.1 : Durée de vie par régime thermique (15 min)

La durée de vie moyenne dépend-elle du régime thermique ? Sur le tableau des organes, regroupez par le niveau de sollicitation thermique et calculez, pour chaque régime, l'effectif, la moyenne et l'écart-type de `RUL_jours`. Écrivez le tableau de ces résultats dans un nouveau classeur Excel.

**Résultat attendu.** Régime faible : 226,4 jours sur 119 organes, écart-type de 34,2 jours ; régime moyen : 300,6 jours sur 49 organes, écart-type de 15,6 jours ; régime élevé : 90,9 jours sur 69 organes, écart-type de 60,0 jours.

**Seconde voie.** La moyenne d'un régime se recalcule par un masque : on garde les organes du régime puis on prend la moyenne de leur durée de vie.

**Indication.** Après `groupby(niveau)`, la colonne `RUL_jours` reçoit `agg` avec les trois fonctions `count`, `mean` et `std` : un tableau à trois colonnes et une ligne par régime. La méthode `to_excel` l'écrit dans un classeur. Le régime élevé a la durée de vie moyenne la plus courte, avec le plus fort écart-type.

Les contrôles de la cellule lisent la variable `moyennes` : gardez ce nom.

In [ ]:
# Votre code :
assert abs(moyennes["Faible"] - 226.4) < 0.05, "régime faible : 226,4 jours attendus"
assert abs(moyennes["Moyen"] - 300.6) < 0.05, "régime moyen : 300,6 jours attendus"
assert abs(moyennes["Élevé"] - 90.9) < 0.05, "régime élevé : 90,9 jours attendus"
print("Exercice 8.1 : contrôles passés")

### Section 8.2 : un tableau croisé met les effectifs en regard.

In [ ]:
modes = pd.read_excel(CHEMIN_MODES)
colonnes_modes = [c for c in modes.columns if c.startswith("Panne_")]
nb_modes = modes[colonnes_modes].sum(axis=1)
# notion: croise
croise = pd.crosstab(modes["Panne"], nb_modes, margins=True)
croise_part = pd.crosstab(modes["Panne"], nb_modes, normalize="index")
print(croise)
assert croise.loc["All", "All"] == 10000 and croise.loc[1, 0] == 9, "9 pannes sans mode"
assert abs(croise_part.loc[0].sum() - 1.0) < 1e-12, "normalize='index' : chaque ligne somme à un"

In [ ]:
# notion: pivot
flotte_courte = flotte[["ID_Machine", "Temp_Carter_K", "Pression_Ligne_bar", "Vib_RMS_AxeZ_mm_s", "Courant_Moteur_Amp"]]
moyennes_machine = flotte_courte.pivot_table(index="ID_Machine", aggfunc="mean")
print("tableau pivot des moyennes par machine :", moyennes_machine.shape)
assert moyennes_machine.shape == (100, 4), "100 machines, 4 grandeurs"
long = modes.melt(id_vars="time", value_vars=colonnes_modes, var_name="mode", value_name="actif")
assert long.shape == (50000, 3), "melt empile les cinq colonnes de modes : 5 x 10 000 lignes"

In [ ]:
# notion: recoder
codes = {"Faible": 0, "Moyen": 1, "Élevé": 2}
niveau_code = df[niveau].map(codes)
assert niveau_code.isna().sum() == 3 and niveau_code.max() == 2
couple_kNm = df["couple_charge_Nm"].div(1000)
puissance_W = df["puissance_mecanique_kw"].mul(1000)
ecarts = (df["couple_charge_Nm"] - df["couple_charge_Nm"].mean()).abs()
assert abs(couple_kNm.iloc[0] * 1000 - df["couple_charge_Nm"].iloc[0]) < 1e-9
assert (
    abs(puissance_W.iloc[0] / 1000 - df["puissance_mecanique_kw"].iloc[0]) < 1e-9 and ecarts.min() >= 0
)
assert df["RUL_jours"].round(-2).iloc[0] in (100, 200, 0, 300)
dictionnaire = bilan["effectif"].to_dict()
assert dictionnaire == {"Faible": 119, "Moyen": 49, "Élevé": 69}

## Exercice 8.2 : Pannes par mode de panne (15 min)

Dans le fichier des modes de panne, empilez les cinq colonnes de modes en une seule avec `melt`, en gardant la colonne `time` et en désignant les cinq colonnes à empiler par l'argument `value_vars`. Additionnez ensuite les indicateurs de chaque mode avec `groupby`. Quel mode est le plus fréquent ? Combien de lignes compte le tableau long ?

**Résultat attendu.** Outil : 46 ; hydraulique : 115 ; puissance : 95 ; surcharge : 98 ; alimentation : 19 ; total 373 ; tableau long de 50 000 lignes.

**Seconde voie.** La somme directe des cinq colonnes du tableau d'origine donne les mêmes totaux, sans `melt`.

**Indication.** La méthode `melt` reçoit `id_vars="time"` et la liste des cinq colonnes de modes ; elle rend un tableau de $5 \times 10\,000$ lignes. Un `groupby` sur la colonne des modes, suivi de la somme des indicateurs, additionne chaque mode. Le mode hydraulique est le plus fréquent.

Les contrôles de la cellule lisent la variable `pannes_par_mode` : gardez ce nom.

In [ ]:
# Votre code :
assert (
    pannes_par_mode.tolist() == [46, 115, 95, 98, 19]
), "pannes par mode attendues : 46, 115, 95, 98, 19"
assert (
    pannes_par_mode.equals(modes[colonnes_modes].sum().astype(pannes_par_mode.dtype))
), "melt ou somme directe"
print("Exercice 8.2 : contrôles passés")

### Section 8.3 : une fusion se valide avant de s'employer.

In [ ]:
pannes = pd.read_excel(CHEMIN_PANNES)
# notion: merge
gauche = pd.DataFrame({"cle": [1, 2, 3], "a": [10, 20, 30]})
droite = pd.DataFrame({"cle": [2, 3, 4], "b": [200, 300, 400]})
tailles = {how: len(pd.merge(gauche, droite, on="cle", how=how)) for how in ("inner", "left", "right", "outer")}
print("lignes selon le type de jointure :", tailles)
assert tailles == {"inner": 2, "left": 3, "right": 3, "outer": 4}

In [ ]:
# notion: merge-valide
doublons = pd.DataFrame({"cle": [2, 2, 3], "b": [1, 2, 3]})

try:
    pd.merge(gauche, doublons, on="cle", validate="one_to_one")
    verdict = "fusion acceptée"
except pd.errors.MergeError as erreur:
    verdict = "fusion refusée : clé en double à droite"

print(verdict)
assert (
    verdict.startswith("fusion refusée")
), "validate arrête une fusion qui dupliquerait des lignes"
marquee = pd.merge(gauche, droite, on="cle", how="outer", indicator=True)
assert (
    marquee["_merge"].astype(str).value_counts().to_dict() == {"both": 2, "left_only": 1, "right_only": 1}
)

In [ ]:
# notion: concat
haut = pd.concat([gauche, gauche], ignore_index=True)
cote = pd.concat([gauche.set_index("cle"), droite.set_index("cle")], axis=1)
assert (
    haut.shape == (6, 2) and cote.shape == (4, 2)
), "empiler les lignes, puis juxtaposer les colonnes"

## Exercice 8.3 : Fusion des deux fichiers de pannes (défi, 25 min)

Le fichier des mesures de la machine tournante (jeu AI4I_maintenance, 10 000 lignes, 9 colonnes) et le fichier des modes de panne (7 colonnes) partagent la colonne `time`. Fusionnez-les sur cette clé en exigeant une correspondance une à une, sans perdre ni dupliquer une ligne. La colonne `Panne` existe des deux côtés : pandas la suffixe. Vérifiez qu'elle dit la même chose des deux côtés, et contrôlez les dimensions du résultat.

**Résultat attendu.** 10 000 lignes et 15 colonnes ($9+ 7- 1$ : la clé n'est comptée qu'une fois) ; colonnes `Panne_x` et `Panne_y` identiques ; aucune ligne d'un seul côté.

**Seconde voie.** `indicator=True` ajoute une colonne `_merge` qui vaut « both » sur toutes les lignes, et `validate` n'a levé aucune erreur.

**Indication.** La fonction `pd.merge` reçoit les deux tableaux, `on="time"`, `how="outer"`, `validate="one_to_one"` et `indicator=True` : elle rend 10 000 lignes et, avec la colonne `_merge`, une colonne de plus que 15. Les colonnes `Panne_x` et `Panne_y` se comparent par `==`, suivi de `all()`. Toutes les valeurs de `_merge` valent `both` : aucune clé ne manque d'un côté. On retire `_merge` par `drop(columns="_merge")` une fois la vérification faite.

Les contrôles de la cellule lisent les variables `doublon_panne`, `fusion` : gardez ces noms.

In [ ]:
# Votre code :
assert (
    fusion.shape[0] == 10000 and fusion.shape[1] == 9 + 7 - 1 + 1
), "10 000 lignes, 15 colonnes et le marqueur"
assert (fusion["_merge"].astype(str) == "both").all(), "aucune ligne orpheline"
assert (
    (fusion["Panne_mesure"] == fusion["Panne_mode"]).all()
), "la colonne partagée dit la même chose"
assert doublon_panne == ["Panne_mesure", "Panne_mode"]
# Votre code :
assert fusion.shape == (10000, 15), "dimensions attendues : 10000 x 15"
print("Exercice 8.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F08-1 : Regroupements (`pandas.DataFrame.groupby`)

In [ ]:
import numpy as np
import pandas as pd

tirants = pd.DataFrame({"nuance": ["T4", "T6", "T4", "T6", "T4"],
                        "k_N_m": [1.0e6, 2.0e6, 1.2e6, 2.2e6, 1.1e6]})
moyennes = tirants.groupby("nuance")["k_N_m"].mean()
verif = (1.0e6 + 1.2e6 + 1.1e6) / 3
assert np.isclose(moyennes["T4"], verif)  # contrôle pour le groupe T4
msg = f"T4 : {moyennes['T4']/1e6:.1f} MN/m, T6 : {moyennes['T6']/1e6:.1f} MN/m"
print(msg.replace(".", ","))

### Fiche F08-2 : Jointures (`pandas.merge`)

In [ ]:
import numpy as np
import pandas as pd

geom = pd.DataFrame({"arbre": [10, 11, 12, 13], "d_mm": [20, 25, 20, 30]})
rupt = pd.DataFrame({"arbre": [12, 10, 11], "C_Nm": [850, 810, 1620]})
fusion = pd.merge(geom, rupt, on="arbre", how="inner", validate="one_to_one")

tau = 16 * fusion["C_Nm"] / (np.pi * (fusion["d_mm"] * 1e-3)**3)
texte = " ".join(f"{t*1e-6:.1f}" for t in tau).replace(".", ",")
print(f"Contraintes de cisaillement : {texte} MPa")
test = np.isclose(fusion["C_Nm"].iloc[0], 810.0)  # Arbre 10 en tête
print(f"Alignement réussi sur la première ligne : {test}")